### Check the raw data

In [ ]:
import os

data_dir = "data/raw"
for category in os.listdir(data_dir):
    files = os.listdir(f"{data_dir}/{category}")
    print(f"{category}: {len(files)} files")
    count_not_mp3 = 0
    for file in files:
        if (file.split(".")[1] != 'mp3'):
            count_not_mp3 += 1
    if count_not_mp3 != 0:
        print(f"\033[91{count_not_mp3} files are of different type")
    else:
        print("\033[92All files are mp3\033[0m")

### Contamination Check Script

In [ ]:
import os
import numpy as np
import librosa
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

def extract_features(filepath):
    try:
        y, sr = librosa.load(filepath, sr=22050, mono=True, duration=10)
        
        # Mel spectrogram summary
        mel = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=128)
        mel_db = librosa.power_to_db(mel, ref=np.max)
        
        # Feature vector: stats over time axis
        features = np.concatenate([
            np.mean(mel_db, axis=1),
            np.std(mel_db, axis=1),
            np.mean(librosa.feature.mfcc(y=y, sr=sr, n_mfcc=20), axis=1),
            np.std(librosa.feature.mfcc(y=y, sr=sr, n_mfcc=20), axis=1),
            [librosa.feature.spectral_centroid(y=y, sr=sr).mean()],
            [librosa.feature.spectral_rolloff(y=y, sr=sr).mean()],
            [librosa.feature.zero_crossing_rate(y).mean()]
        ])
        return features
    except Exception as e:
        print(f"Error loading {filepath}: {e}")
        return None

def load_folder(folder_path):
    features, filenames = [], []
    files = [f for f in os.listdir(folder_path) if f.endswith(('.mp3', '.wav', '.ogg'))]
    
    for i, fname in enumerate(files):
        print(f"Processing {i+1}/{len(files)}: {fname}", end='\r')
        feat = extract_features(os.path.join(folder_path, fname))
        if feat is not None:
            features.append(feat)
            filenames.append(fname)
    
    return np.array(features), filenames

def find_optimal_clusters(features_scaled, max_k=8):
    inertias, silhouettes = [], []
    k_range = range(2, max_k + 1)
    
    for k in k_range:
        km = KMeans(n_clusters=k, random_state=42, n_init=10)
        labels = km.fit_predict(features_scaled)
        inertias.append(km.inertia_)
        silhouettes.append(silhouette_score(features_scaled, labels))
    
    # Plot elbow + silhouette
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    
    ax1.plot(k_range, inertias, 'bo-')
    ax1.set_xlabel('Number of clusters (k)')
    ax1.set_ylabel('Inertia')
    ax1.set_title('Elbow Method')
    
    ax2.plot(k_range, silhouettes, 'ro-')
    ax2.set_xlabel('Number of clusters (k)')
    ax2.set_ylabel('Silhouette Score')
    ax2.set_title('Silhouette Score (higher = better)')
    
    plt.tight_layout()
    plt.savefig('cluster_analysis.png', dpi=150)
    plt.show()
    
    best_k = k_range[np.argmax(silhouettes)]
    print(f"\nOptimal k by silhouette: {best_k}")
    return best_k

def cluster_and_visualize(features_scaled, filenames, k):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(features_scaled)
    
    # PCA to 2D for visualization
    pca = PCA(n_components=2)
    reduced = pca.fit_transform(features_scaled)
    
    plt.figure(figsize=(10, 7))
    scatter = plt.scatter(reduced[:, 0], reduced[:, 1], c=labels, cmap='tab10', alpha=0.7)
    plt.colorbar(scatter, label='Cluster')
    plt.title(f'Audio Clusters (k={k})')
    plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)')
    plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)')
    plt.savefig('clusters_pca.png', dpi=150)
    plt.show()
    
    # Print cluster contents
    for cluster_id in range(k):
        cluster_files = [filenames[i] for i, l in enumerate(labels) if l == cluster_id]
        print(f"\nCluster {cluster_id}: {len(cluster_files)} files")
        for f in cluster_files[:5]:  # show first 5
            print(f"  {f}")
        if len(cluster_files) > 5:
            print(f"  ... and {len(cluster_files) - 5} more")
    
    return labels


In [ ]:

data_dir = "data/raw"
for category in os.listdir(data_dir):
    folder = f"data/raw/{category}"  # change this
    print(f"\n\n\033[91mProcessing {category}\033[0m\n\n")
    print("Extracting features...")
    features, filenames = load_folder(folder)
    print(f"\nLoaded {len(features)} files")

    scaler = StandardScaler()
    features_scaled = scaler.fit_transform(features)

    print("\nFinding optimal clusters...")
    best_k = find_optimal_clusters(features_scaled)

    print(f"\nClustering with k={best_k}...")
    labels = cluster_and_visualize(features_scaled, filenames, best_k)

### Load audio and make plot

In [ ]:
"""
Plots 10 random audio samples as MEL Spectrogram from each category.
"""
import os
import librosa
import librosa.display
import numpy as np
import matplotlib.pyplot as plt
import random

# Parameters
SR = 22050
WINDOW_SIZE = 3
WINDOW_SAMPLES = SR * WINDOW_SIZE  # 66150
HOP_LENGTH = 512
N_FFT = 2048
N_MELS = 128
OVERLAP = 0.5

def process_audio_to_mel_spec(file_path):
    try:

        y, _ = librosa.load(file_path, sr=SR)
        
        to_melspectrogram(y)
        
    except Exception as e:
        print(f"❌ Error processing {file_path}: {e}")
        return None, None

        
def plot_spectrogram(mel, sample_rate, title="Mel-Spectrogram", ax=None):
    """
    Helper function to visualize the 2D spectrogram.
    """
    

    img = librosa.display.specshow(
        mel,
        sr=sample_rate,
        hop_length=512,
        x_axis='time',
        y_axis='mel',
        cmap='viridis',
        ax=ax
    )

    ax.set_title(title, fontsize=8)

    return img
# --- Execution Example ---

if __name__ == "__main__":

    all_categories = os.listdir("./data/raw")
    n_categories = len(all_categories)

    # Big figure containing everything
    fig, axes = plt.subplots(
        n_categories,
        10,
        figsize=(25, 4 * n_categories),
        constrained_layout=True
        )

    # Handle single-category case
    if n_categories == 1:
        axes = [axes]

    for row, category in enumerate(all_categories):

        files = os.listdir(f"./data/raw/{category}")
        sampled_files = random.sample(files, min(10, len(files)))

        for col, id in enumerate(sampled_files):

            SAMPLE_FILE = f"./data/raw/{category}/{id}"

            if os.path.exists(SAMPLE_FILE):

                # print(f"Processing: {SAMPLE_FILE}")

                mel = process_audio_to_mel_spec(SAMPLE_FILE)

                if mel is not None:

                    img = plot_spectrogram(
                        mel,
                        SR,
                        title=os.path.basename(SAMPLE_FILE),
                        ax=axes[row][col]
                    )

        # Category label on left side
        axes[row][0].set_ylabel(category, fontsize=12)

    # One shared colorbar for entire figure
    fig.colorbar(
        img,
        ax=axes,
        format='%+2.0f dB',
        shrink=0.6,
        pad=0.02
    )

    # plt.tight_layout()
    plt.show()

### Script for converting raw data to processed data

In [ ]:
"""
This scripts converts the raw data in to the processed data
"""
import os
import numpy as np
import librosa

# Parameters
SR = 22050
WINDOW_SIZE = 3
WINDOW_SAMPLES = SR * WINDOW_SIZE  # 66150
HOP_LENGTH = 512
N_FFT = 2048
N_MELS = 128
OVERLAP = 0.5

CONTINUOUS = ["wind", "rain", "birds", "crickets", "thunder"]
IMPULSIVE = ["gunshots", "glass_breaking", "chainsaw"]

LABELS = {
    "wind": 0, "rain": 1, "birds": 2,
    "crickets": 3, "thunder": 4, "gunshots": 5,
    "glass_breaking": 6, "chainsaw": 7
}

def to_melspectrogram(window):
    """ 
    Returns a 2-D mel spectrogram converted to db
    """
    mel = librosa.feature.melspectrogram(
        y=window,
        sr=SR,
        n_mels=N_MELS,
        n_fft=N_FFT,
        hop_length=HOP_LENGTH
    )
    mel_db = librosa.power_to_db(mel, ref=np.max)
    return mel_db

def process_continuous(y):
    """Slide windows with 50% overlap, no trimming."""
    samples = []
    step = int(WINDOW_SAMPLES * (1 - OVERLAP))
    
    for start in range(0, len(y) - WINDOW_SAMPLES + 1, step):
        window = y[start:start + WINDOW_SAMPLES]
        mel = to_melspectrogram(window)
        samples.append(mel)
    
    return samples

def augment_window(window):
    """Adds augmented audio sample by adding background noise to original"""
    augmented = [window]  # keeps original
    
    # Add background noise
    noise = np.random.normal(0, 0.005, len(window)).astype(np.float32)
    augmented.append(window + noise)
    return augmented

def process_impulsive(y):
    """Trim silence, center window around peak energy."""
    # Trim leading/trailing silence
    y_trimmed, _ = librosa.effects.trim(y, top_db=20)
    
    if len(y_trimmed) < WINDOW_SAMPLES:
        # Pad if trimmed audio is shorter than window
        y_trimmed = np.pad(y_trimmed, (0, WINDOW_SAMPLES - len(y_trimmed)))
        windows = augment_window(y_trimmed[:WINDOW_SAMPLES])
        return [to_melspectrogram(w) for w in windows]
    
    # Find peak energy frame
    energy = np.array([
        np.sum(y_trimmed[i:i+1024]**2)
        for i in range(0, len(y_trimmed) - 1024, 512)
    ])
    peak_frame = np.argmax(energy)
    peak_sample = peak_frame * 512
    
    # Center window around peak
    start = max(0, peak_sample - WINDOW_SAMPLES // 2)
    end = start + WINDOW_SAMPLES
    
    # Adjust if window goes out of bounds
    if end > len(y_trimmed):
        end = len(y_trimmed)
        start = max(0, end - WINDOW_SAMPLES)
    
    window = y_trimmed[start:end]
    
    if len(window) < WINDOW_SAMPLES:
        window = np.pad(window, (0, WINDOW_SAMPLES - len(window)))
    
    windows = augment_window(window)
    return [to_melspectrogram(w) for w in windows]

def process_folder(folder_path, category):
    """Processes all samples of one folder and return list of mel spectrograms of each folder"""
    all_mels = []
    files = [f for f in os.listdir(folder_path) if f.endswith(('.mp3', '.wav', '.ogg'))]
    is_continuous = category in CONTINUOUS
    
    for i, fname in enumerate(files):
        print(f"  {i+1}/{len(files)}: {fname}", end='\r')
        try:
            y, _ = librosa.load(os.path.join(folder_path, fname), sr=SR, mono=True)
            
            if is_continuous:
                mels = process_continuous(y)
            else:
                mels = process_impulsive(y)
            
            all_mels.extend(mels)
        
        except Exception as e:
            print(f"\n  Error on {fname}: {e}")
    
    print(f"\n  {category}: {len(all_mels)} windows extracted")
    return all_mels

def preprocess_all(raw_dir="data/raw", save_dir="data/processed"):
    """Saves mels into disk as numpy arrays"""
    os.makedirs(save_dir, exist_ok=True)
    
    X, y = [], []
    
    for category, label in LABELS.items():
        folder = os.path.join(raw_dir, category)
        if not os.path.exists(folder):
            print(f"Skipping {category} - folder not found")
            continue
        
        print(f"Processing: {category}")
        mels = process_folder(folder, category)
        
        category_dir = os.path.join(save_dir, category)
        os.makedirs(category_dir, exist_ok=True)
        np.save(os.path.join(category_dir, "mels.npy"), np.array(mels))

        X.extend(mels)
        y.extend([label] * len(mels))
    
    X = np.array(X)
    y = np.array(y)
    
    np.save(os.path.join(save_dir, "X.npy"), X)
    np.save(os.path.join(save_dir, "y.npy"), y)
    print(f"\nSaved to {save_dir}/X.npy and {save_dir}/y.npy")
    
    return X, y

if __name__ == "__main__":
    X, y = preprocess_all()

### Check Distribution of the processed data

In [ ]:
""" Checks sample distributiobn of the processed data"""
import numpy as np

X = np.load("data/processed/X.npy")
y = np.load("data/processed/y.npy")

print(f"X shape: {X.shape}")
print(f"X dtype: {X.dtype}")
print(f"X value range: {X.min():.2f} to {X.max():.2f}")

LABELS = {
    0: "wind", 1: "rain", 2: "birds",
    3: "crickets", 4: "thunder", 5: "gunshots",
    6: "glass_breaking", 7: "chainsaw"
}

print("\nSamples per category:")
for label, name in LABELS.items():
    count = np.sum(y == label)
    print(f"  {name}: {count}")

## First model architechture

In [ ]:
from typing import Any

import torch
import torch.nn as nn

class Conv2D(nn.Module):
    def __init__(self, in_channels, out_channels, *args: Any, **kwargs: Any) -> None:
        super().__init__(*args, **kwargs)
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )
    def forward(self, x):
        return self.block(x)
    
class CNN(nn.Module):
    def __init__(self, num_classes: int= 8, dropout: float = 0.5, *args: Any, **kwargs: Any) -> None:
        super().__init__(*args, **kwargs)
        self.features = nn.Sequential(
            Conv2D(1, 32),
            Conv2D(32, 64),
            Conv2D(64, 128),
            Conv2D(128, 256)
        )
        self.pool = nn.AdaptiveAvgPool2d((1,1))
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.pool(x)
        x = x.flatten(start_dim=1)
        return self.classifier(x)
    
    

In [ ]:
model = CNN()
dummy = torch.randn(4,1,128,130)
out = model(dummy)
out.shape

In [ ]:
total = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total params:     {total:,}")
print(f"Trainable params: {trainable:,}")

In [ ]:
import numpy as np
import torch
from torch.utils.data import Dataset

class AcousticDataset(Dataset):
    def __init__(self, X:np.ndarray, y:np.ndarray) -> None:
        self.X = torch.tensor(X, dtype=torch.float32).unsqueeze(1)
        self.y = torch.tensor(y, dtype=torch.long)
    
    def __len__(self):
        return len(self.y)
    
    def __getitem__(self, idx) -> Any:
        return self.X[idx], self.y[idx]
    

In [ ]:
X = np.load("../data/processed/X.npy")
y = np.load("../data/processed/y.npy")
dataset = AcousticDataset(X,y)
len(dataset)

In [ ]:
sample_X, sample_y = dataset[0]
print(sample_X.shape)
print(sample_y)
print(sample_X.dtype)
print(sample_y.dtype)

In [ ]:
torch.cuda.is_available()

In [ ]:
torch.cuda.is_available()

In [ ]:
import librosa
y, window_samples = librosa.load("..//data/raw/gunshots/20352.mp3", sr=22050, mono=True)


In [ ]:
y

In [ ]:
y.shape[0]


In [ ]:
for start in range(0, y.shape[0] - window_samples + 1, window_samples):
    print(start)